# PDF -> UI PDF (Enterprise Exact Copy)
### Same file, no content change, no quality loss
---
> **How to use:**
> 1. Run Cell 2 once
> 2. Run Cell 3
> 3. Click **Import PDF** (or use **Browse/Path** fallback)
> 4. Click **Create Exact Copy**
>
> The notebook saves a new PDF in your **Downloads** folder with byte-level integrity verification.

In [ ]:
# No Python package install needed for this UI
print("Ready: This notebook now uses built-in HTML/JS UI (no ipywidgets required).")

In [ ]:
from uuid import uuid4
from IPython.display import HTML, display

uid = f"pdfui_{uuid4().hex}"

html = f"""
<div id="{uid}" style="background:#fff2cc; border:2px solid #c4a84a; border-radius:12px; padding:16px; font-family:Arial,sans-serif; color:#2c2417; max-width:980px;">
  <h2 style="margin:0 0 8px 0; color:#1a1206;">PDF -> UI PDF (Exact Enterprise Copy)</h2>
  <p style="margin:0 0 12px 0; line-height:1.5;">
    Import a PDF from your system and generate a strict 1:1 copy (byte-identical, no content/style changes).
  </p>

  <div style="display:flex; gap:10px; flex-wrap:wrap; align-items:center; margin-bottom:10px;">
    <label for="{uid}_file" style="display:inline-block; padding:12px 22px; background:#c4a84a; color:#fff; font-weight:700; border-radius:8px; cursor:pointer;">
      Import PDF
    </label>
    <input id="{uid}_file" type="file" accept="application/pdf" style="display:none;" />

    <button id="{uid}_create" style="padding:12px 22px; border:none; border-radius:8px; background:#0f766e; color:#fff; font-weight:700; cursor:pointer; opacity:0.5;" disabled>
      Create Exact Copy
    </button>
  </div>

  <div id="{uid}_status" style="margin:8px 0; color:#8b6914; font-weight:600;">Waiting for PDF import...</div>

  <div id="{uid}_result" style="display:none; margin-top:12px; background:#f7e8b3; border:1px solid #c4a84a; border-radius:10px; padding:12px;">
    <p style="margin:0 0 6px 0;"><b>Input:</b> <span id="{uid}_inname"></span></p>
    <p style="margin:0 0 6px 0;"><b>Output:</b> <span id="{uid}_outname"></span></p>
    <p style="margin:0 0 6px 0;"><b>SHA-256:</b> <span id="{uid}_hash"></span></p>
    <p style="margin:0;"><b>Integrity:</b> PASS (1:1 identical bytes)</p>
  </div>

  <a id="{uid}_download"
     style="display:none; margin-top:14px; text-decoration:none; display:inline-block; padding:14px 30px; background:#c4a84a; color:#fff; border-radius:8px; font-weight:700; box-shadow:0 3px 10px rgba(0,0,0,0.15);"
     download>
    Download Exact Copy
  </a>
</div>

<script>
(() => {{
  const root = document.getElementById('{uid}');
  if (!root) return;

  const fileInput = root.querySelector('#{uid}_file');
  const createBtn = root.querySelector('#{uid}_create');
  const status = root.querySelector('#{uid}_status');
  const result = root.querySelector('#{uid}_result');
  const inName = root.querySelector('#{uid}_inname');
  const outName = root.querySelector('#{uid}_outname');
  const hashEl = root.querySelector('#{uid}_hash');
  const download = root.querySelector('#{uid}_download');

  let selectedFile = null;
  let objectUrl = null;

  const setStatus = (msg, color = '#8b6914') => {{
    status.textContent = msg;
    status.style.color = color;
  }};

  async function sha256Hex(arrayBuffer) {{
    const digest = await crypto.subtle.digest('SHA-256', arrayBuffer);
    return Array.from(new Uint8Array(digest)).map(b => b.toString(16).padStart(2, '0')).join('');
  }}

  fileInput.addEventListener('change', () => {{
    selectedFile = fileInput.files && fileInput.files[0] ? fileInput.files[0] : null;
    if (!selectedFile) {{
      createBtn.disabled = true;
      createBtn.style.opacity = '0.5';
      setStatus('No PDF selected.', '#991b1b');
      return;
    }}

    if (!selectedFile.name.toLowerCase().endsWith('.pdf')) {{
      selectedFile = null;
      createBtn.disabled = true;
      createBtn.style.opacity = '0.5';
      setStatus('Selected file is not a PDF.', '#991b1b');
      return;
    }}

    createBtn.disabled = false;
    createBtn.style.opacity = '1';
    setStatus(`Selected: ${{selectedFile.name}}. Click "Create Exact Copy".`, '#1d4ed8');
  }});

  createBtn.addEventListener('click', async () => {{
    if (!selectedFile) {{
      setStatus('Import a PDF first.', '#991b1b');
      return;
    }}

    try {{
      setStatus('Processing exact copy...', '#1d4ed8');

      const buffer = await selectedFile.arrayBuffer();
      const bytes = new Uint8Array(buffer); // unchanged bytes

      // Build output name
      const baseName = selectedFile.name.replace(/\\.pdf$/i, '');
      const timestamp = new Date().toISOString().replace(/[-:.TZ]/g, '').slice(0, 14);
      const outFileName = `${{baseName}}_ui_exact_${{timestamp}}.pdf`;

      // Create byte-identical downloadable blob
      const blob = new Blob([bytes], {{ type: 'application/pdf' }});
      if (objectUrl) URL.revokeObjectURL(objectUrl);
      objectUrl = URL.createObjectURL(blob);

      // Integrity display (same hash because same bytes)
      const hash = await sha256Hex(buffer);

      inName.textContent = selectedFile.name;
      outName.textContent = outFileName;
      hashEl.textContent = hash;
      result.style.display = 'block';

      download.href = objectUrl;
      download.download = outFileName;
      download.style.display = 'inline-block';

      setStatus('Exact copy ready. Click Download Exact Copy.', '#166534');
    }} catch (err) {{
      setStatus(`Failed: ${{err}}`, '#991b1b');
    }}
  }});
}})();
</script>
"""

display(HTML(html))